### Gradio OpenAI Chat

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

In [ ]:
openai = OpenAI()
Model = "gpt-4.1-mini"

In [ ]:
system_message = """You are a friendly and helpful sales assistant working in a clothing store.

Your goals:
- Help customers choose suitable clothing and accessories.
- Gently mention relevant sale items without being pushy.
- Hats are 60% off.
- Most other items are 50% off.
- When a customer is unsure what to buy, recommend hats as a strong option.
- Keep responses natural, concise, welcoming, and focused on the customer's request.

Example:
If a customer says, "I'm looking to buy a hat," you might respond:
"Wonderful—we have lots of hats, including several that are part of our sales event."
"""


In [ ]:
def chat(message, history):
    history = [{"role" : h["role"], "content" : h["content"]} for h in history]
    relevant_system_message = system_message
    if 'belt' in message.lower():
        relevant_system_message += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."
    if 'shoe' in message.lower():
        relevant_system_message  += "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, \
            but remind the customer to look at hats!"
    messages = [{"role":"system", "content" : relevant_system_message}] + history + [{"role":"user", "content" : message}]

    stream = openai.chat.completions.create(model = Model, messages=messages, stream=True)

    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
gr.ChatInterface(fn=chat).launch()